In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP & ALIGNED DATA INGESTION
# ==============================================================================
import os
import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Initializing Experiment C: Recent Tree-Based Architectures...")

# 1. Path Configuration
# NOTE FOR REVIEWERS: Set `base_path` to the repository root directory.
base_path = '..'
source_data_path = f'{base_path}/artifacts/03_Experiment_B_Hybrid_Local/all_predictions.csv'
export_dir = f'{base_path}/artifacts/03_Experiment_C_Recent_Architectures'
os.makedirs(export_dir, exist_ok=True)

# 2. Strict Row-Aligned Data Ingestion
# We load the exact dataset used in Experiment B to ensure perfect row alignment for Notebook 5's Paired Bootstrap Test
df = pd.read_csv(source_data_path)
y_true = df['actual_normalized_score'].values
texts = df['clean_essay_text'].tolist()

# Extract the explicitly engineered pseudo-traits
trait_cols = [
    'trait_1_mechanical_density', 'trait_2_casing_density',
    'trait_3_lexical_richness', 'trait_4_prompt_relevance', 'trait_5_word_count'
]
X_traits = df[trait_cols].values

print(f"Successfully loaded {len(texts)} essays and aligned pseudo-traits.")

Initializing Experiment C: Recent Tree-Based Architectures...
Successfully loaded 308 essays and aligned pseudo-traits.


In [4]:
# ==============================================================================
# FROZEN BERT SEMANTIC EXTRACTION
# ==============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Extracting Dense Semantic Vectors via {device.type.upper()}...")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()

embeddings = []
batch_size = 16

with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="Processing BERT [CLS] Batches"):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts, return_tensors='pt', truncation=True,
            max_length=512, padding=True
        ).to(device)

        outputs = bert_model(**inputs)
        # Extract the [CLS] token representing sequence-level semantics
        cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_emb)

# 2. Construct the Hybrid Feature Matrix
# 768 (Semantic Vectors) + 5 (Explicit Traits) = 773 Dimensional Space
X_hybrid = np.hstack((np.vstack(embeddings), X_traits))

print(f"Final Hybrid Feature Matrix Shape: {X_hybrid.shape}")

Extracting Dense Semantic Vectors via CPU...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing BERT [CLS] Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Final Hybrid Feature Matrix Shape: (308, 773)


In [5]:
# ==============================================================================
# TREE-BASED ARCHITECTURE EXECUTION (OOF PREDICTIONS)
# ==============================================================================
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

print("Executing 5-Fold Cross-Validation for Tree-Based Baselines...")

# 1. Strict CV Configuration (Seed locked for reproducibility)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Architecture 1: Frozen BERT + XGBoost + Pseudo-Traits
print("Training Model 1: Frozen BERT + XGBoost...")
xgb_base = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    tree_method='hist'
)
xgb_oof_preds = cross_val_predict(xgb_base, X_hybrid, y_true, cv=kf, n_jobs=-1)

# 3. Architecture 2: Frozen BERT + PCA + LwXGBoost
# Compresses the 768 BERT dimensions down to 100 core components before boosting
print("Training Model 2: PCA Compressed BERT + LwXGBoost...")
pca_xgb = Pipeline([
    ('pca', PCA(n_components=100, random_state=42)),
    ('xgb', XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        tree_method='hist'
    ))
])
pca_xgb_oof_preds = cross_val_predict(pca_xgb, X_hybrid, y_true, cv=kf, n_jobs=-1)

print("Out-of-Fold predictions successfully generated for both architectures.")

Executing 5-Fold Cross-Validation for Tree-Based Baselines...
Training Model 1: Frozen BERT + XGBoost...
Training Model 2: PCA Compressed BERT + LwXGBoost...
Out-of-Fold predictions successfully generated for both architectures.


In [6]:
# ==============================================================================
# ARTIFACT EXPORT & METRICS SANITY CHECK
# ==============================================================================
from sklearn.metrics import cohen_kappa_score

print("==================================================")
print("EXPERIMENT C: OOF SANITY CHECK METRICS")
print("==================================================")

def calculate_qwk(y_true_cont, y_pred_cont):
    # Enforce identical 11-point discretization (0-10) used across the pipeline
    y_t_int = np.round(y_true_cont * 10).astype(int)
    y_p_int = np.round(y_pred_cont * 10).astype(int)
    return cohen_kappa_score(y_t_int, y_p_int, weights='quadratic')

# Calculate sanity check metrics
# (Official comparative calculations will happen in Notebook 05)
qwk_xgb = calculate_qwk(y_true, xgb_oof_preds)
qwk_pca = calculate_qwk(y_true, pca_xgb_oof_preds)

print(f"Frozen BERT + XGBoost QWK     : {qwk_xgb:.4f}")
print(f"Frozen BERT + PCA + LwXGBoost : {qwk_pca:.4f}")
print("==================================================")

# Export Artifacts for Notebook 05 Interoperability
xgb_df = pd.DataFrame({
    'xgb_oof_preds': xgb_oof_preds,
    'pca_xgb_oof_preds': pca_xgb_oof_preds
})

export_path = f'{export_dir}/xgboost_predictions.csv'
xgb_df.to_csv(export_path, index=False)
print(f"\nArtifacts successfully exported to: {export_path}")
print("Notebook 03_Experiment_C_Recent_Architectures is now COMPLETE.")

EXPERIMENT C: OOF SANITY CHECK METRICS
Frozen BERT + XGBoost QWK     : 0.6516
Frozen BERT + PCA + LwXGBoost : 0.6543

Artifacts successfully exported to: /content/drive/MyDrive/AES_Files/Artifacts/03_Experiment_C_Recent_Architectures/xgboost_predictions.csv
Notebook 03_Experiment_C_Recent_Architectures is now COMPLETE.
